# Task 4.2 — Predict HAI Magnitude for H3N2 A/Massachusetts/18/2022 (D28)

**4.2 predict magnitude of antibody response - H3N2 A/Massachusetts/18/2022 (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI / Measure: Single strain titer / Metric: Spearman correlation
* Full description: HAI titer for H3N2 A/Massachusetts/18/2022 at Day 28

---

## Design notes

**Proxy target:** The target strain (H3N2 A/Massachusetts/18/2022) is absent from the training data.
We approximate it by averaging the Day 28 HAI titers of all other H3N2 strains present in the dataset.
All HAI columns are then dropped to prevent leakage.

**Spearman correlation:** ranks predictions and truth; rewards monotonic agreement regardless of scale.
Robust to outliers. Score: 1.0 = perfect, 0.0 = no signal, -1.0 = reversed.

**5-fold cross-validation:** each participant's prediction is made by a model that never saw them during training.

In [ ]:
TARGET_COL = 'H3N2_proxy_d28'  # proxy: target strain absent from training data
AUTO_ML_MAX_RUNTIME_SECONDS = 60 * 30

In [ ]:
CSV_PATH = '../merged_data/combined.csv'
CHALLENGE_DATA_PATH = '../cleaned_data'
SUBMISSION_PATH = '../automl_submission'

In [ ]:
import io
import os
import warnings
from contextlib import redirect_stderr, redirect_stdout

import h2o
import numpy as np
import pandas as pd
from h2o.automl import H2OAutoML
from scipy.stats import spearmanr

In [ ]:
challenge_participants = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_participants_cleaned.csv')
challenge_hai = pd.read_csv(CHALLENGE_DATA_PATH + '/challenge_hai_cleaned.csv')
challenge_tran = pd.read_csv(CHALLENGE_DATA_PATH + '/transcriptomics_challenge_pca.csv')
challenge_data = challenge_hai.merge(challenge_participants, on='participant_id', how='inner')
challenge_data = challenge_data.merge(challenge_tran, on='participant_id', how='left')

# Add same HAI aggregate features as training
ch_h3n2_d0 = [c for c in challenge_data.columns if 'H3N2' in c and c.endswith('_d0')]
ch_h1n1_d0 = [c for c in challenge_data.columns if 'H1N1' in c and c.endswith('_d0')]
ch_all_d0   = [c for c in challenge_data.columns if c.startswith('HAI_') and c.endswith('_d0')]

challenge_data['HAI_mean_H3N2_d0']  = challenge_data[ch_h3n2_d0].mean(axis=1)
challenge_data['HAI_mean_H1N1_d0']  = challenge_data[ch_h1n1_d0].mean(axis=1)
challenge_data['HAI_mean_all_d0']   = challenge_data[ch_all_d0].mean(axis=1)
challenge_data['HAI_n_measured_d0'] = challenge_data[ch_all_d0].notna().sum(axis=1)

print(f'Challenge shape: {challenge_data.shape}')

### Preprocessing — proxy target\n\nThe target strain is absent from the training set. We create a proxy by averaging all H3N2 Day 28 HAI titers.\nRows are filtered to those with a valid proxy target, then all-null columns are removed on the filtered subset.\nThe `_d28`/`_d365` columns are excluded from features at training time to prevent leakage.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Raw shape: {df.shape}')

In [ ]:
# Missing data summary by feature group
missing = df.isna().mean().rename('pct_missing')
missing = missing[missing > 0].sort_values(ascending=False)

def feature_group(col):
    if col.startswith('TRAN_'): return 'TRAN'
    if col.startswith('HAI_'):  return 'HAI'
    if col.startswith('PART_'): return 'PART'
    return 'other'

summary = missing.to_frame()
summary['group'] = summary.index.map(feature_group)
summary['n_missing'] = (missing * len(df)).astype(int)

print("Missing data by group:")
print(summary.groupby('group')['pct_missing'].agg(['count', 'min', 'max', 'mean']).round(3))
print(f"\nTotal columns with any missing: {len(missing)} / {df.shape[1]}")
print(f"\nTop 20 most-missing columns:")
display(summary.head(20))

In [ ]:
# Create proxy target: row-wise average of all H3N2 d28 HAI strains
h3n2_d28_cols = [c for c in df.columns if c.startswith('HAI_') and 'H3N2' in c and c.endswith('_d28')]
df[TARGET_COL] = df[h3n2_d28_cols].mean(axis=1)
print(f'Proxy target averaged from {len(h3n2_d28_cols)} H3N2 d28 strains')

In [ ]:
# Keep only participants with transcriptomics data
df = df[df['TRAN_PC1'].notna()]
print(f'Rows with transcriptomics: {len(df)}')

# Drop columns that are all-null in this subset
all_null_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=all_null_cols)
print(f'Dropped {len(all_null_cols)} all-null columns → {df.shape[1]} remaining')

# Drop d0 columns that are >50% missing — single-cohort strains not worth imputing
sparse_d0 = [c for c in df.columns if c.endswith('_d0') and df[c].isna().mean() > 0.5]
df = df.drop(columns=sparse_d0)
print(f'Dropped {len(sparse_d0)} sparse d0 columns → {df.shape[1]} remaining')

# Filter to rows with a valid proxy target
df = df[df[TARGET_COL].notna()]
print(f'Rows with valid proxy target: {len(df)}')

In [ ]:
print(f'Shape: {df.shape}')
print(f'Target stats:\n{df[TARGET_COL].describe()}')
print(f'\ndtype counts:\n{df.dtypes.value_counts()}')
print(f'\nMissing per column (top 10):\n{df.isna().sum().sort_values(ascending=False).head(10)}')

---
## AutoML Setup

In [ ]:
warnings.filterwarnings('ignore', category=UserWarning, module='h2o')
h2o.init()

In [ ]:
print(f'Converting to H2O: {df.shape[0]} rows × {df.shape[1]} columns')
col_types = {c: 'real' for c in df.select_dtypes(include=['number']).columns}
data = h2o.H2OFrame(df, column_types=col_types)
print(f'H2OFrame shape: {data.shape}')

In [ ]:
h3n2_d0 = [c for c in df.columns if 'H3N2' in c and c.endswith('_d0')]
h1n1_d0 = [c for c in df.columns if 'H1N1' in c and c.endswith('_d0')]
all_d0   = [c for c in df.columns if c.startswith('HAI_') and c.endswith('_d0')]

df['HAI_mean_H3N2_d0']  = df[h3n2_d0].mean(axis=1)
df['HAI_mean_H1N1_d0']  = df[h1n1_d0].mean(axis=1)
df['HAI_mean_all_d0']   = df[all_d0].mean(axis=1)
df['HAI_n_measured_d0'] = df[all_d0].notna().sum(axis=1)
print(f'Added 4 HAI aggregate features. Shape: {df.shape}')

---
## AutoML Training

In [ ]:
# Features: everything available at d0 — exclude all d28/d365 targets and participant_id
x = [c for c in data.columns
     if not c.endswith('_d28') and not c.endswith('_d365')
     and c != 'participant_id']
y = TARGET_COL

train = data[data[y].isna() == 0]
print(f'Training samples: {train.nrows}  |  Features: {len(x)}')

aml = H2OAutoML(
    seed=1,
    nfolds=5,
    keep_cross_validation_predictions=True,
    max_runtime_secs=AUTO_ML_MAX_RUNTIME_SECONDS,
    exclude_algos=['DeepLearning', 'GLM'],
)

_buf = io.StringIO()
with redirect_stdout(_buf), redirect_stderr(_buf):
    aml.train(x=x, y=y, training_frame=train)
print('Training complete.')

In [ ]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

In [ ]:
# Stacked Ensembles don't store CV predictions — fall back to best base model if needed
model = aml.leader
if model.cross_validation_holdout_predictions() is None:
    for m_id in aml.leaderboard['model_id'].as_data_frame()['model_id']:
        m = h2o.get_model(m_id)
        if m.cross_validation_holdout_predictions() is not None:
            model = m
            break

cv_preds = model.cross_validation_holdout_predictions().as_data_frame()['predict']
actuals = train[y].as_data_frame()[y]
rho, pval = spearmanr(actuals, cv_preds)
print(f'Task 4.2 — Spearman (5-fold CV): {rho:.3f}  (p={pval:.4f})')
print(f'Model used for eval: {model.model_id}')

In [ ]:
print(f'Leader model: {aml.leader.model_id}')
print(f'Varimp model: {model.model_id}')
varimp = model.varimp(use_pandas=True)
display(varimp.head(20))
model.varimp_plot(num_of_features=20)

In [ ]:
challenge_col_types = {c: 'real' for c in challenge_data.select_dtypes(include=['number']).columns}
challenge_hf = h2o.H2OFrame(challenge_data, column_types=challenge_col_types)

y_pred = aml.leader.predict(challenge_hf).as_data_frame()['predict']

os.makedirs(SUBMISSION_PATH, exist_ok=True)
results = pd.DataFrame({
    'Participant_ID': challenge_data['participant_id'].values,
    'Task_4.2': np.exp2(y_pred),
})
results.to_csv(f'{SUBMISSION_PATH}/task_4_2.csv', index=False)
results

In [ ]:
h2o.cluster().shutdown()

---
## Conclusion

- **Leader model:** (fill after run)
- **CV Spearman:** (fill after run)

**Target:** Proxy HAI titer for H3N2 A/Massachusetts/18/2022 at D28 (averaged from all available H3N2 d28 strains).
The target strain is absent from the training data; this proxy is our best approximation.

Submission saved to `automl_submission/task_4_2.csv` (raw titer scale via `np.exp2`).